# Serve MusPsy on Qwen3.5-9B via vLLM on RunPod (RTX 5090 / Blackwell)

Fork of `muspsy_serve_vllm.ipynb`, specifically for the Qwen3.5-9B backbone on an RTX 5090. Kept as a **separate notebook** rather than adding a third `BASE_MODEL_CHOICE` option, because Qwen3.5-9B needs a genuinely different install (newer vLLM, torch 2.8+/cu128 for Blackwell support) and a different serving strategy (see Step 2) — mixing that into the main notebook's `vllm==0.9.2`/`torch==2.7.0` pin (which serves Llama-3/3.1-8B on an older-generation GPU and should stay untouched) would risk both.

**Why Qwen3.5-9B was previously abandoned, and what changed:**
1. **VRAM** — on a 24GB card, weights + hybrid-GDN state cache + `torch.compile` overhead alone consumed 23.30 GiB, leaving ~0 room for KV cache (`torch.OutOfMemoryError` even for one request). An RTX 5090 (32GB) should have real headroom for this now — untested at this specific card's numbers, so watch VRAM on first launch.
2. **`--enable-lora` silently no-ops on Qwen3.5/3.6's hybrid-GDN architecture** ([vllm-project/vllm#49354](https://github.com/vllm-project/vllm/issues/49354), still open as of this writing) — vLLM serves the un-fine-tuned base model with **no error**. A workaround is documented in that issue's comments: `merge_and_unload()` the LoRA adapter into a dense checkpoint, then serve that directly *without* `--enable-lora` (reported ~0.97 output-match parity vs. HF PEFT). **This notebook's `SERVE_MODE="muspsy"` therefore expects a merged checkpoint, not a LoRA adapter** — see Step 2. That merge step doesn't exist yet in `muspsy_finetune.ipynb`; it needs to be added there before this notebook's `muspsy` mode has anything real to point at.
3. **Qwen3.5-9B is a vision-language model** (`Qwen3_5ForConditionalGeneration`, `pipeline_tag: image-text-to-text`), not pure text. `--trust-remote-code` is needed; this reproduction only ever uses it as a text-only chat endpoint.

**Toggle**: `SERVE_MODE` below is `"muspsy"` (merged fine-tuned checkpoint) or `"base"` (plain Qwen3.5-9B, no adapter — the fairness-comparison condition). Only one server runs at a time; stop (Step 8) before switching and re-launching (Step 4).

**Cost note**: the pod bills by the hour whether or not you're sending requests — stop the server (and the pod) when done testing.

## Step 0: Check CUDA works before installing anything
Cheap to check now (seconds), expensive to discover after a multi-minute install. This session hit the same failure repeatedly on certain Community Cloud pods: `nvidia-smi` reports a perfectly healthy GPU, but `torch.cuda.is_available()` returns `False` — traced to `CUDA_VISIBLE_DEVICES` being set to an empty string, which hides all devices from the CUDA runtime (a different code path than `nvidia-smi`'s, which uses NVML and isn't affected). Secure Cloud pods resolved it reliably; if this check fails, try a different pod before spending time on Step 1.

In [ ]:
!nvidia-smi

try:
    import torch
    print("torch already installed:", torch.__version__)
    print("cuda available:", torch.cuda.is_available())
    assert torch.cuda.is_available(), (
        "CUDA not available even though nvidia-smi above may show a healthy GPU. If "
        "CUDA_VISIBLE_DEVICES is set to an empty string, that's the cause (echo it to check). "
        "Try a different pod - Secure Cloud resolved this reliably - before proceeding to Step 1."
    )
    print("CUDA check passed - safe to proceed to Step 1.")
except ImportError:
    print("torch not installed yet on this pod - will be installed fresh in Step 1 "
          "(`uv pip install vllm --torch-backend=auto`). Re-run this cell after Step 1 "
          "to confirm CUDA works before launching the server.")

## Step 1: Install vLLM (torch==2.11.0/cu129, for Blackwell)

Two version constraints stack here, both non-negotiable:
- **RTX 5090 (Blackwell, `sm_120`) needs torch 2.8+** — earlier torch releases have no Blackwell support at all, matching the same reasoning already applied in `muspsy_finetune.ipynb`'s Step 2.
- **vLLM's default PyPI wheels have a confirmed packaging bug** ([vllm-project/vllm#43435](https://github.com/vllm-project/vllm/issues/43435), closed "not planned"): the compiled `_C_stable_libtorch` extension references `libcudart.so.13` regardless of `--torch-backend`. The fix (found in that issue's comments after closure, and already validated once in this project for the Qwen3.5 compatibility test): install from vLLM's own CUDA-specific wheel index instead of default PyPI.

`vllm==0.25.1` is pinned here, installed via the `wheels.vllm.ai` cu129 index (which hosts only the `vllm` wheel itself, not torch). `torch`/`torchvision`/`torchaudio` are pinned to `2.11.0`/`0.26.0`/`2.11.0` — the exact versions `vllm==0.25.1` itself declares as dependencies (confirmed on this pod via the diagnostic print in the cell below, not guessed from GitHub source browsing), installed from `download.pytorch.org/whl/cu129`. A real run on this pod confirmed these exact versions import cleanly (no ABI `undefined symbol` error) - an earlier `ImportError: undefined symbol: torch_list_size` at `import vllm` turned out to be a leftover broken install from a prior attempt, not a genuinely wrong version pin.

`--index-strategy unsafe-best-match` is required: without it, uv's default "first-index-only" strategy finds an old `packaging<=24.1` on the pytorch cu129 index and won't fall through to PyPI for the newer `packaging>=24.2` that vllm's own `flashinfer-python==0.6.13` dependency needs, and resolution fails outright ("No solution found") before anything installs. The explicit torch/torchvision/torchaudio pins above are what keep that package family locked to the exact right build even under `unsafe-best-match` searching all indexes for everything else.

In [ ]:
!pip install -U uv --break-system-packages
# vllm==0.25.1 via vLLM's own CUDA-specific wheel index (not default PyPI - see
# markdown above for the libcudart.so.13 packaging bug this dodges).
#
# torch/torchvision/torchaudio ARE pinned explicitly here, to the exact versions
# vllm==0.25.1 itself declares (confirmed via the diagnostic print below, from a
# real run on this pod - not guessed from GitHub source this time: torch==2.11.0,
# torchvision==0.26.0, torchaudio==2.11.0). That same run also proved these exact
# versions DO import cleanly (no ABI "undefined symbol" error) once actually
# installed - the earlier ImportError was never really about the version pin being
# wrong, it was almost certainly a leftover broken install from a prior attempt
# that hadn't been cleaned up.
#
# --index-strategy unsafe-best-match IS needed (re-added after briefly removing
# it): without it, uv's default "first-index-only" strategy finds SOME "packaging"
# package on the pytorch cu129 index (an old <=24.1 build) and refuses to look at
# PyPI for a newer one, even though vllm's own dependency flashinfer-python==0.6.13
# requires packaging>=24.2 - producing "No solution found when resolving
# dependencies" before anything gets installed. unsafe-best-match lets uv fall
# through to PyPI for packages like `packaging` that aren't torch-family. The
# explicit torch/torchvision/torchaudio pins above are what keep THAT package
# family pinned to the exact right build even under unsafe-best-match, instead of
# letting the resolver wander for them the way the very first (unpinned) attempt did.
#
# --break-system-packages: this pod's Python 3.12 base image marks itself
# "externally managed" (PEP 668), which blocks --system installs by default -
# a newer-base-image behavior not seen on the earlier (pre-Blackwell) pods this
# project used. Safe to override here: the pod is an ephemeral, single-purpose
# container with no separate system Python environment worth protecting. Applied
# to the plain `pip install -U uv` above too, not just the `uv pip install --system`
# call below - on a fresh pod without uv pre-installed, that line would hit the
# identical block (it only worked without the flag once, when uv already happened
# to be present and pip's "Requirement already satisfied" check short-circuited
# before it needed to write anything).
!uv pip install --system --break-system-packages --reinstall \
    "vllm==0.25.1" "torch==2.11.0" "torchvision==0.26.0" "torchaudio==2.11.0" \
    --extra-index-url https://wheels.vllm.ai/0.25.1/cu129 \
    --extra-index-url https://download.pytorch.org/whl/cu129 \
    --index-strategy unsafe-best-match

# Ground truth, logged BEFORE the import that would crash on a real mismatch: what
# torch requirement vllm actually declares, and what got resolved for it.
import importlib.metadata as _im
print("vllm declares these torch-related requirements:")
for _req in _im.requires("vllm") or []:
    if "torch" in _req.lower():
        print(" ", _req)
print("torch actually resolved/installed:", _im.version("torch"))

import torch, vllm
print("torch:", torch.__version__)
print("torch cuda:", torch.version.cuda)
print("cuda available:", torch.cuda.is_available())
print("vllm:", vllm.__version__)
assert torch.cuda.is_available(), "CUDA not available - check nvidia-smi and that this pod actually has a Blackwell (RTX 5090) GPU attached."

assert torch.__version__.split("+")[0] == "2.11.0", (
    f"torch pin failed, got {torch.__version__!r} (expected 2.11.0, the exact version "
    f"vllm==0.25.1 itself declares - see the diagnostic print above). If this happens "
    f"right after a pod Stop/Edit/Start cycle, the container disk was likely reset - "
    f"this cell is safe to re-run from scratch."
)
assert vllm.__version__ == "0.25.1", f"vllm pin failed, got {vllm.__version__!r}"

# The actual regression test for vllm-project/vllm#43435: importing vllm is where
# the libcudart.so.13 error surfaced before (vllm.platforms.cuda imports
# vllm._C_stable_libtorch at import time). Reaching this print without an
# ImportError confirms the wheel-index fix worked.
print("\n=== vLLM imported successfully - libcudart issue did not reproduce. ===")

## Step 2: Choose what to serve

`"muspsy"` — the fine-tuned counselor, served from a **merged dense checkpoint** (base weights + LoRA adapter already combined via `merge_and_unload()`). `MERGED_CHECKPOINT` below must point at an already-merged repo/path - see `muspsy_finetune.ipynb`'s Step 5c for the merge step.

`"base"` — plain Qwen3.5-9B, no fine-tuning (the fairness-comparison condition: same backbone your framework's baselines would use).

`"muspsy_unmerged"` — serves the LoRA adapter directly via `--enable-lora`/`--lora-modules`, no merge involved. **Note on why this exists**: this was originally built to test whether [vllm-project/vllm#49354](https://github.com/vllm-project/vllm/issues/49354) (a reported silent no-op of `--enable-lora` on Qwen3.5's hybrid-GDN architecture) reproduces here - a real test made it LOOK confirmed (Task 1/Task 2 output didn't match the trained format at all), which is why the merge step got built. That test turned out to be invalid: it was missing `chat_template_kwargs: {"enable_thinking": false}`, so what actually got returned was the model's un-terminated thinking trace, cut off by `max_tokens` - not a sign the adapter wasn't applied. Once Step 6b was fixed to include that flag, **the merged checkpoint produced a near-perfect match to ground truth** - but so would the unmerged path have, in all likelihood, since the actual root cause was never about LoRA application at all. `#49354` was never actually confirmed to reproduce on this build; the merge step turned out not to be strictly necessary here, though it's not wasted - you have a working merged checkpoint either way. `LORA_ADAPTER_PATH` below can be a local path or an HF Hub repo ID.

`--trust-remote-code` is on unconditionally — Qwen3.5-9B is a vision-language model (`Qwen3_5ForConditionalGeneration`), and this notebook only ever exercises it as a text-only chat endpoint.

In [ ]:
# Qwen3.5-9B backbone (fixed - this notebook is Qwen3.5-9B-only, see intro; the
# main muspsy_serve_vllm.ipynb keeps handling llama3-8b/qwen3-8b on the older-GPU
# tier separately).
BASE_MODEL = "Qwen/Qwen3.5-9B"

SERVE_MODE = "muspsy"  # "muspsy", "base", or "muspsy_unmerged" (diagnostic - see Step 2 markdown)
# Must be an already-MERGED checkpoint (base + LoRA combined via merge_and_unload()),
# NOT the bare LoRA adapter repo - see Step 2 markdown for why (--enable-lora silently
# no-ops on Qwen3.5's hybrid-GDN architecture, vllm-project/vllm#49354). Placeholder
# repo ID below - muspsy_finetune.ipynb doesn't have the merge step yet that would
# produce this; update once that exists.
MERGED_CHECKPOINT = "thanaphatt1/qwen3.5-9b-muspsy-fixed-merged"  # HF Hub repo ID, or a local path

# Only used when SERVE_MODE == "muspsy_unmerged" - the raw LoRA adapter (NOT merged),
# served via --enable-lora as a direct test of whether vllm-project/vllm#49354 actually
# reproduces on this exact vllm==0.25.1 build. Local path if this pod also ran
# muspsy_finetune.ipynb's training (e.g. "../LLaMA-Factory/saves/qwen3.5-9b/lora/muspsy-fixed"),
# or the HF Hub repo id if it was pushed there instead.
LORA_ADAPTER_PATH = "thanaphatt1/qwen3.5-9b-muspsy-fixed"  # local path or HF Hub repo ID

API_PORT = 8000
API_KEY = "muspsy-dev-key"  # shared bearer token thesis-framework will send - change this to something private

# Conservative starting point for an untested card/model combination - the 5090's
# 32GB should have real headroom past what caused the earlier 24GB-card OOM (weights
# + hybrid-GDN state cache + torch.compile overhead alone consumed 23.30 GiB there),
# but that hasn't been measured on this specific card yet. Start conservative, watch
# actual VRAM usage in Step 5's server log, and raise once a real number is known -
# this 6144 is just the same starting point the main notebook uses for its
# LoRA-protected "muspsy" mode, not derived from anything Qwen3.5-9B-specific.
max_model_len = 6144

assert SERVE_MODE in ("muspsy", "base", "muspsy_unmerged")

if SERVE_MODE == "muspsy":
    _model_arg = MERGED_CHECKPOINT
elif SERVE_MODE == "muspsy_unmerged":
    _model_arg = BASE_MODEL  # base weights; --lora-modules layers the adapter on top below
else:
    _model_arg = BASE_MODEL

serve_cmd = [
    "vllm", "serve",
    _model_arg,
    "--port", str(API_PORT),
    "--api-key", API_KEY,
    "--max-model-len", str(max_model_len),
    "--gpu-memory-utilization", "0.85",
    "--dtype", "bfloat16",
    "--trust-remote-code",  # Qwen3.5-9B is a VLM (Qwen3_5ForConditionalGeneration) - see Step 2.
    # Works around a real crash: vLLM's profile_run() synthesizes a dummy image input to
    # size the encoder cache (log line: "Encoder cache will be initialized with a budget of
    # 16384 tokens, and profiled with 1 image items..."), which runs the vision tower's
    # forward pass through vLLM's bundled FlashAttention-2 kernel (vllm_flash_attn /
    # _vllm_fa2_C). That kernel is a known, documented gap on Blackwell/SM100+
    # (vllm-project/vllm#38411: "Vision encoder crashes on SM100 ... FA2 compiled for
    # SM80-only, no override available for vision encoder") - it doesn't support this
    # architecture yet, surfacing here as a misleading "CUDA error: the provided PTX was
    # compiled with an unsupported toolchain" rather than a clear "unsupported arch" error.
    # This reproduction is 100% text-only and never sends images, so telling vLLM up front
    # that 0 images/videos are allowed per prompt should skip that dummy vision forward pass
    # (and the encoder-cache sizing that needs it) entirely, since there's nothing to profile
    # capacity for. Cheap to test either way - the crash happens within seconds of launch.
    # Confirmed to work on a real run: this fix got the server all the way past the crash
    # and on to LoRA loading (see --max-lora-rank below for what happened next).
    "--limit-mm-per-prompt", '{"image": 0, "video": 0}',
]

if SERVE_MODE == "muspsy_unmerged":
    # Diagnostic path only - see Step 2 markdown. This is exactly the pattern #49354
    # reports as silently no-op'ing on Qwen3.5's hybrid-GDN layers; that's the thing
    # being tested here, not assumed to work.
    #
    # --max-lora-rank 32: vLLM's own default cap is 16, and our adapter was trained with
    # lora_rank: 32 (muspsy_finetune.ipynb's config) - without raising this, vLLM correctly
    # and LOUDLY rejects loading the adapter at all ("ValueError: LoRA rank 32 is greater
    # than max_lora_rank 16"), before ever getting to test whether #49354 itself reproduces.
    # This is a real, separate bug from #49354 - a plain missing CLI flag, not a silent
    # no-op - confirmed by the explicit ValueError (the #49354 bug produces no error at all).
    serve_cmd += [
        "--enable-lora", "--max-lora-rank", "32",
        "--lora-modules", f"muspsy={LORA_ADAPTER_PATH}",
    ]

# served_model_name must match thesis-framework/utils/llm_utils.py's get_llm() vllm:
# branch, which expects "muspsy" or "<backbone>-base" (get_llm("vllm:muspsy") /
# get_llm("vllm:qwen3.5-9b-base")) - --served-model-name aliases the model to a short
# name instead of forcing API requests to use its full HF Hub / local path.
if SERVE_MODE in ("muspsy", "muspsy_unmerged"):
    served_model_name = "muspsy"
elif SERVE_MODE == "base":
    served_model_name = "qwen3.5-9b-base"
else:
    raise ValueError(f"Unknown SERVE_MODE: {SERVE_MODE!r}")
serve_cmd += ["--served-model-name", served_model_name]

print(f"SERVE_MODE={SERVE_MODE!r}, serving {_model_arg!r}")
if SERVE_MODE == "muspsy_unmerged":
    print(f"LoRA adapter (unmerged, diagnostic): {LORA_ADAPTER_PATH!r}")
print(f"served model name for API requests: {served_model_name!r}")
print(f"max_model_len: {max_model_len}")
print("Command:", " ".join(serve_cmd))

## Step 3: Set your Hugging Face token
Qwen/Qwen3.5-9B isn't gated the way Meta-Llama-3-8B-Instruct is, but a token is still needed for `SERVE_MODE="muspsy"` if `MERGED_CHECKPOINT` points at a private Hub repo.

In [ ]:
%env HF_TOKEN=hf_your_read_or_write_token_here

## Step 4: Launch the server in the background
`vllm serve` blocks forever while serving, so it's launched as a background process here (not with a blocking `!` cell) — this lets you keep using the notebook to health-check and test it, and to stop it cleanly later.

In [ ]:
import subprocess, os

env = os.environ.copy()
# vLLM's internal EngineCore worker process defaults to fork(), which fails with
# "Cannot re-initialize CUDA in forked subprocess" whenever CUDA has already been
# touched in the parent process before the fork happens. spawn starts a fresh
# interpreter instead, avoiding the inherited CUDA context.
env["VLLM_WORKER_MULTIPROC_METHOD"] = "spawn"

# Works around a real crash hit on this exact pod: the profiling/warmup run got all
# the way through model loading (17.66 GiB) and torch.compile successfully, then
# crashed in the sampler with `RuntimeError: FlashInfer requires GPUs with sm75 or
# higher` - misleading on an RTX 5090 (sm_120, which trivially clears sm75).
# flashinfer's check_cuda_arch() only fails that check when its detected arch list
# is EMPTY (any major>=8 alone passes), which happens when its torch.cuda-based
# auto-detection silently fails inside the EngineCore worker and gets swallowed by
# a try/except that logs a warning and continues with nothing detected - so the
# error is actually about failed arch *detection* in that subprocess, not the GPU
# being too old. Rather than chase why detection fails in that specific spawned
# worker, side-step it entirely: this forces vLLM's PyTorch-native top-k/top-p
# sampler instead of JIT-compiling FlashInfer's kernel. Slightly slower sampling,
# functionally identical output - fine for this reproduction's request volumes.
env["VLLM_USE_FLASHINFER_SAMPLER"] = "0"

log_file = open("vllm_server.log", "w")
server_proc = subprocess.Popen(
    serve_cmd,
    stdout=log_file,
    stderr=subprocess.STDOUT,
    env=env,
)
print(f"Server starting in background, PID={server_proc.pid}. Logs: vllm_server.log")

## Step 5: Wait for the server to become ready
Loading the 8B model + vLLM engine init typically takes a couple of minutes. This polls until the `/v1/models` endpoint responds.

In [ ]:
import time, requests

url = f"http://localhost:{API_PORT}/v1/models"
headers = {"Authorization": f"Bearer {API_KEY}"}

for attempt in range(60):
    if server_proc.poll() is not None:
        raise RuntimeError(
            f"Server process exited early (code {server_proc.returncode}). Check vllm_server.log for the traceback."
        )
    try:
        resp = requests.get(url, headers=headers, timeout=5)
        if resp.status_code == 200:
            print("Server is up.")
            print(resp.json())
            break
    except requests.exceptions.ConnectionError:
        pass
    time.sleep(5)
else:
    raise TimeoutError("Server did not become ready in time. Check vllm_server.log.")

## Step 6: Sanity-check with a sample request
Confirms the OpenAI-compatible chat endpoint actually works before wiring up `thesis-framework`. Note the `model` field must be `served_model_name` from Step 2 (`"muspsy"` for the LoRA adapter, or the exact base model ID for `SERVE_MODE="base"`) - vLLM's OpenAI server matches requests against the served model name(s), not an arbitrary string.

In [ ]:
chat_url = f"http://localhost:{API_PORT}/v1/chat/completions"
payload = {
    "model": served_model_name,
    "messages": [
        {"role": "user", "content": "Hi, I've been feeling anxious about an upcoming exam. Can you help?"}
    ],
    "temperature": 0.7,
    "max_tokens": 300,
}
resp = requests.post(chat_url, headers={**headers, "Content-Type": "application/json"}, json=payload, timeout=60)
resp.raise_for_status()
print(resp.json()["choices"][0]["message"]["content"])

## Step 6b: Test Task 1 and Task 2 format against real training data
Step 6 only exercises generic chat, not MusPsy's own Task 1 (memory extraction) or Task 2 (goal planning) format - the two tasks that actually broke on the Qwen3-8B backbone earlier in this project (Task 1 collapsed to a bare `Counseling Summary:` with none of its required bracketed sections). This pulls real records directly from MusPsy's own `train/task1.json`/`task2.json` (the exact `instruction`+`input` fields the adapter was actually trained on - not a reimplementation like `run_muspsy_simulation.py`'s own `TASK1_INSTRUCTION`/`TASK2_INSTRUCTION`, which are this project's separate prompt text for real PsychEval simulations and aren't guaranteed byte-identical to what's baked into the training data), sends them to whichever `SERVE_MODE` is currently running, and checks the live output against the structural markers real Task 1/Task 2 outputs always contain.

In [ ]:
import requests, json, re

# Real training examples, pulled straight from MusPsy's own repo - not invented prompts.
# Testing against the model's actual training distribution, using the SAME instruction
# text it was fine-tuned on.
TASK1_URL = "https://raw.githubusercontent.com/Rener2005/MusPsy/main/train/task1.json"
TASK2_URL = "https://raw.githubusercontent.com/Rener2005/MusPsy/main/train/task2.json"

task1_records = requests.get(TASK1_URL, timeout=30).json()
task2_records = requests.get(TASK2_URL, timeout=30).json()

# Fixed (not random) index so results are reproducible run-to-run and comparable
# across different SERVE_MODE / backbone tests.
TEST_RECORD_INDEX = 5
task1_example = task1_records[TEST_RECORD_INDEX]
task2_example = task2_records[TEST_RECORD_INDEX]


def _call_muspsy(instruction: str, input_text: str) -> str:
    # LLaMA-Factory's AlpacaDatasetConverter builds the final user-turn content as
    # "\n".join([instruction, input]) - matches run_muspsy_simulation.py's own
    # run_task1_memory_extraction/run_task2_goal_planning convention, and the actual
    # training-time format (task1/2.json's "instruction" already ends in its own "\n",
    # so this produces a blank line between instruction and input, not a plain concat).
    content = "\n".join([instruction, input_text])
    payload = {
        "model": served_model_name,
        "messages": [{"role": "user", "content": content}],
        "temperature": 0.0,  # deterministic - this is a format check, not a diversity/quality test
        "max_tokens": 1024,
        # CRITICAL - without this, Qwen3.5 defaults to thinking mode ON, and since this
        # server isn't started with --reasoning-parser qwen3, the model's internal
        # reasoning trace floods straight into `content` with no separation, getting cut
        # off by max_tokens before it ever reaches the real trained answer. A real run
        # WITHOUT this flag produced rambling "The summary should capture..." meta-
        # commentary that looked exactly like a broken/untrained model - it wasn't the
        # model or the fine-tuning that was broken, it was this test cell missing this
        # one flag. Confirmed directly: identical prompt, only this field added, went
        # from complete format failure to a near-perfect match against ground truth.
        "chat_template_kwargs": {"enable_thinking": False},
    }
    resp = requests.post(chat_url, headers={**headers, "Content-Type": "application/json"}, json=payload, timeout=120)
    resp.raise_for_status()
    return resp.json()["choices"][0]["message"]["content"]


# --- Task 1: memory extraction ---
task1_output = _call_muspsy(task1_example["instruction"], task1_example["input"])
print("=" * 20, "TASK 1 - live output", "=" * 20)
print(task1_output)
print("\n" + "=" * 20, "TASK 1 - ground truth (MusPsy's own training data)", "=" * 20)
print(task1_example["output"])

# Real task1.json output is ALWAYS this exact 5-part shape (verified earlier in this
# project across all 7,182 records): a "The Nth Counseling" title, four bracketed
# sections, then a "Counseling Summary:" block. Collapsing to a bare "Counseling
# Summary:" with none of the bracket sections is exactly the failure mode seen on
# Qwen3-8B - this check catches that immediately instead of requiring a manual read.
task1_markers = [
    "[Emotion and Cognitive State:]",
    "[Counselor Observations:]",
    "[Counseling Assignments:]",
    "[Session Goal:]",
    "Counseling Summary:",
]
print("\n" + "=" * 20, "TASK 1 - structural check", "=" * 20)
for marker in task1_markers:
    print(f"  {'PASS' if marker in task1_output else 'FAIL'}: {marker!r} present")

# --- Task 2: goal planning ---
task2_output = _call_muspsy(task2_example["instruction"], task2_example["input"])
print("\n\n" + "=" * 20, "TASK 2 - live output", "=" * 20)
print(task2_output)
print("\n" + "=" * 20, "TASK 2 - ground truth (MusPsy's own training data)", "=" * 20)
print(task2_example["output"])

# Real task2.json profile blocks use "[Tag]\ntext" (bracket on its own line), not
# "[Tag] text" inline - confirmed against real records earlier in this project, and
# the actual root cause of a severe goal-hallucination/repetition bug when this
# project's own profile-redaction prompt used the wrong (inline) format.
print("\n" + "=" * 20, "TASK 2 - structural check", "=" * 20)
bracket_own_line = re.search(r"\[[A-Za-z ]+\]\n", task2_output)
print(f"  {'PASS' if bracket_own_line else 'FAIL'}: bracket-on-own-line profile format present")

## Step 7: Expose the port to your local machine
In the RunPod dashboard: open this pod's **Connect** panel -> **HTTP Service** (or **TCP Port Mapping** on Secure Cloud) -> expose port `8000`. RunPod gives you a public URL like:

```
https://<pod-id>-8000.proxy.runpod.net
```

From `thesis-framework` on your PC, this is the `base_url` for a new `get_llm()` branch pointed at this server (append `/v1`), with the `API_KEY` set above as the bearer token, and `served_model_name` (`"muspsy"` or the base model ID) as the model string. Requests sent from your PC travel to this URL; all GPU inference happens here on the pod.

## Step 8: Stop the server when done
Run this before switching `SERVE_MODE` and re-launching (Step 4), or before shutting down the pod. Remember the pod itself keeps billing until you stop/terminate it separately in the RunPod dashboard.

In [ ]:
server_proc.terminate()
server_proc.wait(timeout=30)
print("Server stopped.")